# Disk Space Management via Terminal
**A mini-course for macOS**  
**Created:** 2026-05-31  
**Author:** Science Officer Spock — for Lieutenant Commander Data

---

## Why this notebook exists

Your disk got to **91% full** without you noticing.  
This notebook teaches you how to **see what's happening**, **find the culprits**, and **clean safely** — all from the terminal.

## Mental model

```
[Check total]  →  [Find big folders]  →  [Drill deeper]  →  [Delete safely]
    df -h              du -sh *               du -sh folder/*        rm -rf
```

## ⚠️ Golden rule before deleting anything

```
rm -rf is permanent. No Trash. No undo.
Always inspect before you delete.
```

---

## What you will learn

1. Check overall disk usage — `df -h`
2. Find large files and directories — `du + sort`
3. Drill into a specific folder
4. Clean up common space wasters
5. Monitor disk usage live
6. Automate cleanup with a shell script

# Section 1: Check Overall Disk Usage

## Command

```bash
df -h /
```

## What each part means

```
df      →  "disk free" — shows space for mounted filesystems   💾
-h      →  human-readable (GB, MB instead of raw bytes)        👁️
/       →  check the root filesystem (your main drive)         🌳
```

## Reading the output

```
Filesystem     Size    Used   Avail  Capacity
/dev/disk3s1  228Gi   17Gi   21Gi     45%
```

| Column | Meaning |
|--------|---------|
| `Size` | Total disk size |
| `Used` | Space currently used |
| `Avail` | Free space remaining |
| `Capacity` | Percentage used — **watch this number** |

## When to worry

| Capacity | Status |
|----------|--------|
| < 70% | ✅ Healthy |
| 70–85% | ⚠️ Keep an eye on it |
| 85–95% | 🔴 Act now |
| > 95% | 🚨 macOS will slow down — clean immediately |

In [ ]:
import subprocess

result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
print(result.stdout)

# Section 2: Find Large Files and Directories

## Command

```bash
du -sh ~/* 2>/dev/null | sort -rh | head -20
```

## Breaking it down

```
du -sh ~/*      →  "Show size of each item in home folder"    📦
2>/dev/null     →  "Silence permission errors"                🕳️
|               →  "Pipe output to next command"              🚿
sort -rh        →  "Sort by size, largest first"              🔢
|               →  "Pipe again"                               🚿
head -20        →  "Show only top 20 results"                 ✂️
```

## Flags explained

| Flag | Meaning |
|------|---------|
| `du` | disk usage |
| `-s` | summary — one line per item (not recursive breakdown) |
| `-h` | human-readable sizes (GB, MB, KB) |
| `*` | wildcard — match everything inside the folder |
| `sort -r` | reverse order (largest first) |
| `sort -h` | understand human sizes (10G > 500M > 20K) |

## Real example output from this session

```
 30G   ~/Library/Application Support
 22G   ~/Desktop
5.5G   ~/Library/Caches
590M   ~/Downloads
```

→ `Application Support` and `Desktop` were the two big surprises.

In [ ]:
import subprocess
import os

home = os.path.expanduser("~")

# Get sizes of top-level folders in home directory
result = subprocess.run(
    f"du -sh {home}/* 2>/dev/null | sort -rh | head -20",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

# Section 3: Drill Into a Specific Directory

Once you find a large folder in Section 2, you drill deeper — one level at a time.

## Command pattern

```bash
du -sh ~/Library/Application\ Support/* 2>/dev/null | sort -rh | head -20
```

## The `\ ` (backslash space) — escaping spaces in paths

```
Application\ Support   →  tells the terminal: "the space is part of the name, not a separator"
```

Without it:
```bash
du -sh ~/Library/Application Support/*
# Terminal reads: "Application" and "Support/*" as TWO separate arguments → error
```

## Real example from this session — what we found

```
 11G   ~/Library/Application Support/Claude        ← 😱 biggest surprise
6.0G   ~/Library/Application Support/Google
4.5G   ~/Library/Application Support/Code_backup   ← stale backup
2.2G   ~/Library/Application Support/Notion
1.8G   ~/Library/Application Support/Code
1.3G   ~/Library/Application Support/Wispr Flow
1.0G   ~/Library/Application Support/Microsoft Edge
1.0G   ~/Library/Application Support/discord
```

## Then drill one more level — e.g. Claude:

```bash
du -sh ~/Library/Application\ Support/Claude/* 2>/dev/null | sort -rh
```

```
9.7G   Claude/vm_bundles    ← VM images for Claude Code sandbox — safe to delete
749M   Claude/Cache         ← standard cache — safe to delete
236M   Claude/claude-code-vm
```

→ `vm_bundles` alone was **9.7 GB** and regenerates automatically.

In [ ]:
import subprocess

# Change target_folder to wherever you want to drill
target_folder = "~/Library/Application Support"

result = subprocess.run(
    f"du -sh {target_folder}/* 2>/dev/null | sort -rh | head -20",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

# Section 4: Clean Up Common Space Wasters

## ⚠️ Before running any `rm -rf` — always inspect first

```bash
# INSPECT first:
du -sh ~/path/to/folder

# DELETE only after confirming:
rm -rf ~/path/to/folder
```

---

## 4a. Delete a specific folder

```bash
rm -rf ~/Library/Application\ Support/Claude/vm_bundles
```

```
rm      →  remove                                    🗑️
-r      →  recursive (folder + all contents inside)  📂
-f      →  force (no confirmation prompts)           ⚡
```

---

## 4b. Delete everything in a folder EXCEPT specific items

```bash
find ~/Library/Caches -maxdepth 1 -mindepth 1 \
  ! -name "com.openai.atlas" \
  ! -name "com.openai.chat" \
  -exec rm -rf {} +
```

```
find ~/Library/Caches        →  "Search in this folder"
-maxdepth 1 -mindepth 1      →  "Only direct children (not nested)"
! -name "com.openai.chat"    →  "Exclude this item"
-exec rm -rf {} +            →  "Delete all matched items"
```

---

## 4c. Remove Python `__pycache__` folders recursively

```bash
find ~/Desktop/vs_code -type d -name "__pycache__" -exec rm -rf {} + 2>/dev/null
```

---

## 4d. Remove `.DS_Store` files (macOS metadata noise)

```bash
find ~/Desktop -name ".DS_Store" -delete
```

---

## Common targets — what's safe to delete

| Target | Command | Regenerates? |
|--------|---------|--------------|
| Claude `vm_bundles` | `rm -rf ~/Library/Application\ Support/Claude/vm_bundles` | ✅ Yes |
| Browser caches | `rm -rf ~/Library/Caches/Google` | ✅ Yes |
| VS Code backup | `rm -rf ~/Library/Application\ Support/Code_backup` | ✅ Yes (if you still have Code folder) |
| `__pycache__` | `find . -type d -name "__pycache__" -exec rm -rf {} +` | ✅ Yes |
| Notion cache | `rm -rf ~/Library/Application\ Support/Notion` | ✅ Yes — app re-downloads |

In [ ]:
import subprocess

# DRY RUN — preview what __pycache__ folders exist before deleting
# Change the search path to your project folder
search_path = "~/Desktop/vs_code"

result = subprocess.run(
    f"find {search_path} -type d -name '__pycache__' 2>/dev/null",
    shell=True, capture_output=True, text=True
)

folders = result.stdout.strip().split("\n") if result.stdout.strip() else []
print(f"Found {len(folders)} __pycache__ folder(s):\n")
for f in folders:
    print(f"  {f}")

# Section 5: Monitor Disk Usage Over Time

## Take a snapshot — log to file

```bash
df -h / >> ~/Desktop/disk-log.txt
```

```
df -h /   →  check disk usage
>>        →  APPEND to file (not overwrite — note double >>)
```

Run this every time you clean up and you'll have a history.

---

## Live monitoring on macOS (loop in terminal)

```bash
while true; do df -h / | tail -1; sleep 10; done
```

```
while true    →  loop forever
do ... done   →  what to do each iteration
df -h /       →  check disk
| tail -1     →  show only the data line (not the header)
sleep 10      →  wait 10 seconds
```

Press `Ctrl + C` to stop.

---

## Log snapshots with timestamp

```bash
while true; do
  echo "$(date '+%Y-%m-%d %H:%M:%S') — $(df -h / | tail -1)" >> ~/disk-log.txt
  sleep 60
done
```

Output in `disk-log.txt`:
```
2026-05-31 14:05:00 — /dev/disk3s1  228Gi  17Gi  21Gi  45%
2026-05-31 14:06:00 — /dev/disk3s1  228Gi  17Gi  21Gi  45%
```

In [ ]:
import subprocess
from datetime import datetime

# Take a single snapshot and print it
result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
lines = result.stdout.strip().split("\n")
header = lines[0]
data = lines[1]

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Snapshot at {timestamp}")
print(header)
print(data)

# Section 6: Automate Disk Cleanup — Summary Report Script

A reusable script that:
1. Shows current disk usage
2. Finds the largest folders in `~/Library`
3. Finds files older than 30 days in a target folder
4. Prints a clean summary

Run the Python cell below — it executes the script and prints the report.

> **Note:** The script only **reports** — it does not delete anything. Review the output, then decide what to delete manually.

In [ ]:
import subprocess
from datetime import datetime

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("=" * 55)
print(f"  🖥️  DISK REPORT — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 55)

# 1. Overall disk usage
print("\n📊 OVERALL DISK USAGE")
print(run("df -h / | tail -1"))

# 2. Top 10 largest folders in ~/Library
print("\n📦 TOP 10 LARGEST IN ~/Library")
print(run("du -sh ~/Library/* 2>/dev/null | sort -rh | head -10"))

# 3. Top 10 largest folders in ~/Desktop
print("\n🖥️  TOP 10 LARGEST ON ~/Desktop")
print(run("du -sh ~/Desktop/* 2>/dev/null | sort -rh | head -10"))

# 4. Files older than 30 days in ~/Downloads
print("\n📅 FILES OLDER THAN 30 DAYS IN ~/Downloads")
old_files = run("find ~/Downloads -maxdepth 1 -type f -mtime +30 2>/dev/null")
if old_files:
    for line in old_files.split("\n"):
        print(f"  {line}")
else:
    print("  None found.")

print("\n" + "=" * 55)
print("  Review the output above before deleting anything.")
print("=" * 55)